In [5]:
# 03 Machine Learning: Student Performance
# Regression target: G3 (0 to 20). Classification target: passed (G3 >= 10).
# Feature set A = all inputs. Feature set B = without G1 and G2.

In [6]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from aislca.data.ingestion import load_config, load_processed_data
from aislca.data import features
from aislca.ml import train, evaluate


config = load_config()
pass_mark = config["target"]["pass_mark"]
test_size = config["ml"]["test_size"]
random_state = config["ml"]["random_state"]
cv_folds = config["ml"]["cv_folds"]

mat = load_processed_data("mat")
train_df, test_df = features.split_data(mat, pass_mark, test_size, random_state)

print("Train:", len(train_df), "Test:", len(test_df))
print("Pass rate train/test:", train_df["passed"].mean().round(3), test_df["passed"].mean().round(3))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Train: 316 Test: 79
Pass rate train/test: 0.671 0.671


In [7]:
features_A  =  features.get_feature_columns(mat,include_prior_grades=True)
features_B  =  features.get_feature_columns(mat,include_prior_grades=False)
print(len(features_A), len(features_B))



32 30


In [8]:
def run_feature_set(cols, name):
    X_train, X_test = train_df[cols], test_df[cols]
    reg = train.train_models(train.get_regression_models(random_state), X_train, train_df["G3"])
    clf = train.train_models(train.get_classification_models(random_state), X_train, train_df["passed"])
    print(f"===== Feature set {name} =====")
    print("\nRegression (predict G3). Lower MAE/RMSE is better, R2 closer to 1 is better:")
    display(evaluate.compare_models(reg, X_test, test_df["G3"], "regression"))
    print("Classification (predict pass). Higher is better:")
    display(evaluate.compare_models(clf, X_test, test_df["passed"], "classification"))
    return reg, clf

In [14]:
reg_A, clf_A = run_feature_set(features_A, "A (with G1, G2)")

===== Feature set A (with G1, G2) =====

Regression (predict G3). Lower MAE/RMSE is better, R2 closer to 1 is better:


,mae,rmse,r2
Decision Tree,0.880,1.494,0.872
Random Forest,0.952,1.533,0.866
Linear Regression,1.190,1.806,0.813
Baseline (mean),3.164,4.197,-0.008


Classification (predict pass). Higher is better:


,accuracy,precision,recall,f1
Random Forest,0.924,0.943,0.943,0.943
Logistic Regression,0.886,0.879,0.962,0.919
Decision Tree,0.873,0.852,0.981,0.912
Baseline (always pass),0.671,0.671,1.000,0.803


In [16]:
reg_B, clf_B = run_feature_set(features_B, "B (without G1, G2)")

===== Feature set B (without G1, G2) =====

Regression (predict G3). Lower MAE/RMSE is better, R2 closer to 1 is better:


,mae,rmse,r2
Random Forest,2.945,3.789,0.178
Baseline (mean),3.164,4.197,-0.008
Linear Regression,3.365,4.233,-0.026
Decision Tree,3.141,4.336,-0.076


Classification (predict pass). Higher is better:


,accuracy,precision,recall,f1
Decision Tree,0.747,0.770,0.887,0.825
Baseline (always pass),0.671,0.671,1.000,0.803
Random Forest,0.684,0.700,0.925,0.797
Logistic Regression,0.620,0.683,0.811,0.741


In [17]:
X_B = train_df[features_B]
print("Regression, 5-fold cross-validation (lower RMSE is better):")
display(evaluate.cross_validated_scores(
    train.build_pipelines(train.get_regression_models(random_state), X_B),
    X_B, train_df["G3"], "regression", cv_folds, random_state))

print("Classification, 5-fold cross-validation (higher F1 is better):")
display(evaluate.cross_validated_scores(
    train.build_pipelines(train.get_classification_models(random_state), X_B),
    X_B, train_df["passed"], "classification", cv_folds, random_state))

Regression, 5-fold cross-validation (lower RMSE is better):


,rmse_mean,rmse_std
Baseline (mean),4.655,0.403
Linear Regression,4.441,0.469
Decision Tree,4.629,0.329
Random Forest,3.948,0.464


Classification, 5-fold cross-validation (higher F1 is better):


,f1_mean,f1_std
Baseline (always pass),0.803,0.004
Logistic Regression,0.823,0.016
Decision Tree,0.769,0.023
Random Forest,0.805,0.003


In [18]:
evaluate.compare_models(clf_B, test_df[features_B], test_df["passed"],
                        "classification", pos_label=0)

,accuracy,precision,recall,f1
Decision Tree,0.747,0.667,0.462,0.545
Logistic Regression,0.620,0.375,0.231,0.286
Random Forest,0.684,0.556,0.192,0.286
Baseline (always pass),0.671,0.000,0.000,0.000


In [19]:
print("Set A:\n", train.feature_importances(reg_A["Random Forest"]))
print("\nSet B:\n", train.feature_importances(reg_B["Random Forest"]))

Set A:
 num__G2              0.764
num__absences        0.144
num__age             0.012
num__famrel          0.010
num__health          0.006
num__G1              0.005
cat__reason_home     0.004
num__Medu            0.003
num__Fedu            0.003
cat__Mjob_teacher    0.003
dtype: float64

Set B:
 num__absences        0.201
num__failures        0.145
num__goout           0.050
num__health          0.034
num__age             0.034
num__studytime       0.032
num__freetime        0.031
cat__Mjob_at_home    0.031
num__Walc            0.030
num__Medu            0.029
dtype: float64


In [20]:
nonzero_train = train_df[train_df["G3"] > 0]
reg_nz = train.train_models(
    train.get_regression_models(random_state),
    nonzero_train[features_B], nonzero_train["G3"])
evaluate.compare_models(reg_nz, test_df[features_B], test_df["G3"], "regression")

,mae,rmse,r2
Random Forest,2.983,4.073,0.050
Decision Tree,3.102,4.212,-0.016
Baseline (mean),3.243,4.260,-0.039
Linear Regression,3.270,4.285,-0.051
